In [11]:
%pip install pandas numpy scikit-learn matplotlib seaborn wordcloud gensim

Note: you may need to restart the kernel to use updated packages.


# Εξόρυξη Γνώσης από Μουσικά Δεδομένα (Audio & Lyrics)

**Ομάδα:**
* Παναγιώτης Μακαρόνας (AM: sdi2300107)
* Αχιλλέας Νιανιακούδης-Κοέν (AM: sdi2300138)

In [10]:

# Main data handling libraries
import pandas as pd     # Used for dataframes handling and data manipulation
import numpy as np      # Main math library in Python

import tarfile      # Opening tarfile to read its insides
import gensim       # NLP (Natural Language Processing)

# The MLTs (Machine Learning Tools)
from sklearn.cluster import KMeans      # Great for finding clusters and grouping them
from sklearn.decomposition import PCA   # Lowering data size to help the process
from sklearn.manifold import TSNE       # Helping with organizing massive data groups for plots
from sklearn.metrics.pairwise import cosine_similarity  # Finding the similarity between two songs

# Drawing Tools
import matplotlib.pyplot as plt     # Generating plots
import seaborn as sns               # Helps in making the plots more modern
from wordcloud import WordCloud     # Image generator for most frequently used tags in a cluster


## 1. Συλλογή Δεδομένων (Data collection)
Σε αυτό το κελί φορτώνονται τα δεδομένα, εφαρμόζεται το φιλτράρισμα στα Top-5 Genres και πραγματοποιείται το intersection των αρχείων.

In [ ]:
# File paths
genres_path = "data/id_genres.csv"
info_path = "data/id_information.csv"
mfcc_path = "data/id_mfcc_stats.tsv.bz2"
tags_path = "data/id_tags.csv"
lyrics_archive = "data/processed_lyrics.tar.gz"

# Load CSVs, using tab '\t' as a delimiter
df_genres = pd.read_csv(genres_path, sep='\t')
df_info = pd.read_csv(info_path, sep='\t') 
df_tags = pd.read_csv(tags_path, sep='\t')

# Adjusting for compression type as the file is really big
df_mfcc = pd.read_csv(mfcc_path, sep='\t', compression='bz2')

# Top-5 genres filtering
top5 = df_genres['genres'].value_counts().head(5).index.tolist()
df_genres = df_genres[df_genres['genres'].isin(top5)]

# Make sure Id's (due to persisting errors) are strings and strip accidental whitespaces
df_genres['id'] = df_genres['id'].astype(str).str.strip()

# Repeat for MFCCs, assuming the first column is the ID column (which is true in this dataset)
mfcc_id_col = df_mfcc.columns[0]
df_mfcc[mfcc_id_col] = df_mfcc[mfcc_id_col].astype(str).str.strip()

# Find the common IDs between genres and MFCCs and store them
genre_ids = set(df_genres['id'])
mfcc_ids = set(df_mfcc[mfcc_id_col]) 
common_ids = genre_ids & mfcc_ids

# Extract lyrics for the common IDs
lyrics_dict = {}
with tarfile.open(lyrics_archive, 'r:gz') as tar:       # Open the tar.gz file for reading only ('r:gz')

    # For every record in the archive, check if it's a file
    for member in tar:
        if not member.isfile():
            continue
        
        # Extract the Id from the file name (split by '/' splits the path directory and [-1] gets the last name which is the ID) and remove unwanted parts ('.txt' and whitespaces)
        song_id = member.name.split('/')[-1].replace('.txt', '').strip()
        
        # If the id of the song is in our common set, extract the lyrics
        if song_id in common_ids:
            f = tar.extractfile(member) # Read inside the file
            if f:
                # Read and save the files, but decode it using 'utf-8' encoding to avoid unnecessary bytes and errors='replace' which is used to handle
                # any potential decoding issues gracefully by replacing problematic characters with a placeholder
                lyrics_dict[song_id] = f.read().decode('utf-8', errors='replace')

# Construct the final dataset
df_lyrics = pd.DataFrame(list(lyrics_dict.items()), columns=['id', 'lyrics']) # Id and lyrics columns
df = df_genres.merge(df_lyrics, on='id')                                      # Merge genres and lyrics on the 'id' column
df = df.merge(df_mfcc, left_on='id', right_on=mfcc_id_col)                    # Merge the other two with MFCCs, adjusting for their respective 'id' columns

# Now we have way more columns than we need, so we replace the MFCC columns with a single column that contains all the MFCC values as a list
# We create a list of all the MFCC columns excluding the ones we don't need
columns_to_ignore = ['id', 'genres', 'genre', 'lyrics', mfcc_id_col]
mfcc_cols = [col for col in df.columns if col not in columns_to_ignore]

# Compress the MFCC columns into a single list per row
df['mfcc_stats'] = df[mfcc_cols].values.tolist()

# Rename the 'genres' column to 'genre' ('genres' was given by the original CSV files) and create the final dataframe with the necessary columns
df.rename(columns={'genres': 'genre'}, inplace=True)
df_final = df[['id', 'genre', 'lyrics', 'mfcc_stats']]

/tmp/ipykernel_70091/1336543322.py:63: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['mfcc_stats'] = df[mfcc_cols].values.tolist()


## 2. Εξαγωγή Χαρακτηριστικών & Embeddings (Feature Extraction)
Εδώ δημιουργούνται οι διανυσματικές αναπαραστάσεις για το κείμενο (Word2Vec/Doc2Vec) και τον ήχο (PCA στα MFCCs).

In [ ]:

# =============================================
# 2a. TEXT EMBEDDINGS  (Word2Vec → avg vector)
# =============================================

# Tokenize lyrics into word lists
from gensim.models import Word2Vec

tokenized = df['lyrics'].apply(lambda x: x.lower().split()).tolist()

# Train Word2Vec on our corpus (vector_size=128, window=5)
w2v_model = Word2Vec(sentences=tokenized, vector_size=128,
                     window=5, min_count=2, workers=4, epochs=10)
print(f"Word2Vec vocabulary size: {len(w2v_model.wv)}")

# Build a song-level embedding = mean of its word vectors
def song_text_embedding(words, model):
    vecs = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

text_embeddings = np.vstack(
    [song_text_embedding(tok, w2v_model) for tok in tokenized]
)
print(f"Text embeddings shape: {text_embeddings.shape}")  # (N, 128)

# =============================================
# 2b. AUDIO EMBEDDINGS  (PCA on MFCC stats)
# =============================================

# Extract only the numeric MFCC columns (skip id column)
mfcc_cols = [c for c in df.columns if c not in ['id', 'genre', 'lyrics']]
mfcc_matrix = df[mfcc_cols].values.astype(np.float32)

# Replace any NaN/Inf with 0 for safety
mfcc_matrix = np.nan_to_num(mfcc_matrix, nan=0.0, posinf=0.0, neginf=0.0)

# PCA: keep 95% of variance
pca = PCA(n_components=0.95, random_state=42)
audio_embeddings = pca.fit_transform(mfcc_matrix)
print(f"Audio PCA: {mfcc_matrix.shape[1]} dims → {audio_embeddings.shape[1]} dims "
      f"({pca.n_components_} components, 95% variance)")

# Store embeddings back in df for convenience
df['text_emb'] = list(text_embeddings)
df['audio_emb'] = list(audio_embeddings)
print("Embeddings stored in DataFrame ✓")


## 3. Οπτικοποίηση και Ανάλυση (Exploratory Data Analysis - EDA)
Ανάλυση των tags, word clouds, 2D projections (PCA/t-SNE) και Cosine Similarity.

## ΜΕΡΟΣ Β: Κατηγοριοποίηση & Clustering